In [ ]:
from tinydb import TinyDB, Query

from pprint import pprint
from tinydb.operations import delete
# Create a new TinyDB instance (database file will be created in the current directory)

db = TinyDB('gamedata.json')
Object = Query()



In [ ]:

# Insert a sample document if the database is empty
if len(db) == 0:
  db.insert({
    "id":  "66c7c82a8ce9ef0fd008a23e",
    "name": "Earth",
    "shape": "Sphere",
    "mass": 5.972e+24,
    "radius": 6371,
    "color": "#555555",
    "position": {
      "x": 0,
      "y": 0,
      "z": 0
    },
    "velocity": {
        "x": 0,
        "y": 0,
        "z": 0
    },
    "angularVelocity": {
        "wx": 0,
        "wy": 0,
        "wz": 7.2921e-5
    },
    "parent": None,
  })
  db.insert({
    "id":  "66c7c82a8ce9ef0fd008a23f",
    "name": "Moon",
    "shape": "Sphere",
    "mass": 7.347e+22,
    "radius": 6371,
    "color": "#555555",
    "position": {
      "x": 384000,
      "y": 0,
      "z": 0
    },
    "velocity": {
        "x": 0,
        "y": 1.022,
        "z": 0
    },
    "angularVelocity": {
        "wx": 0,
        "wy": 0,
        "wz": 2.6617e-6
    },
    "parent": "66c7c82a8ce9ef0fd008a23e",
  })

# Function to get the object
def get_object():
    Object = Query()
    result = db.search(Object.name.exists())
    if result:
        return result[0]
    else:
        return None

In [ ]:

# Define the query
# Remove the object named "Earth"
#db.remove(Object.name == 'Item')

# for all object name == 'item' add field 'parent' with value '66c7c82a8ce9ef0fd008a23e'
#db.update({'attached': True, "parent":"66c7c82a8ce9ef0fd008a23e"}, Object.name == 'Item')

# print content of the database
print(db.all())


In [ ]:
# Update all entries in the 'objects' table with the name "Item" to add the field "shape" with the value "Pyramid"
db.table('objects').update({'name': 'Resource'}, Object.name == 'Item')

# Print the updated content of the 'objects' table to verify the changes
print(db.table('objects').all())

In [ ]:
# Define the query
Object = Query()

# Get the object with the specified ID
object_with_id = db.search(Object.id == '66c7c82a8ce9ef0fd008a23f')

# Print the object
print(object_with_id)

In [ ]:
import numpy as np

x = np.array([0, 0.4091, 0.9125])
m = 7.2921159e-5
x *= m
xyz_json = {'x': x[0], 'y': x[1], 'z': x[2]}
print(xyz_json)

In [ ]:
import time
print(str(time.time()*1000000))

In [ ]:
import requests
import os

# Define the URL and the local path
url = "https://www.solarsystemscope.com/textures/download/2k_earth_clouds.jpg"
local_path = "dist/textures/" + os.path.basename(url)

# Create the directory if it doesn't exist
os.makedirs(os.path.dirname(local_path), exist_ok=True)

# Download the image and save it to the local path
response = requests.get(url)
with open(local_path, 'wb') as file:
    file.write(response.content)

print(f"Downloaded {url} to {local_path}")

In [ ]:
from tinydb.operations import delete
from pprint import pprint
import numpy as np
import time

# Reference the appropriate table from the TinyDB instance
table = db.table('bodies')

# Get the first body from the bodies table
first_body = table.all()[0]

# Pretty print the fields of the first body
pprint(first_body)

In [ ]:
import shutil

# Copy the gamedata.json file to gamedata._bak
shutil.copy('gamedata.json', 'gamedata._bak')

print("gamedata.json has been copied to gamedata._bak")

In [ ]:
# Generate schema for the items in a table and insert it into the schema table.

#!pip install genson
from genson import SchemaBuilder
from pprint import pprint

table_name = 'resource'
# Get the bodies table from TinyDB
table = db.table(table_name)

builder = SchemaBuilder()

# Generate schema for the bodies table
builder.add_object(table.all())
schema = builder.to_schema()

# Replace 'integer' type with 'number' in the schema
def replace_integer_with_number(schema):
    if isinstance(schema, dict):
        for key, value in schema.items():
            if key == 'type' and value == 'integer':
                schema[key] = 'number'
            else:
                replace_integer_with_number(value)
    elif isinstance(schema, list):
        for item in schema:
            replace_integer_with_number(item)

replace_integer_with_number(schema)
# Insert the generated schema into the schema table
schema_table = db.table('schema')
# Check if the schema for the table already exists
existing_schema = schema_table.search(Object.id == table_name)

if existing_schema:
    # Update the existing schema
    schema_table.update({'schema': schema}, Object.id == table_name)
else:
    # Insert the new schema
    schema_table.insert({'id': table_name, 'schema': schema})

pprint(schema)

In [ ]:
# Drop the 'schema' table
db.drop_table('schema')

# Print the remaining tables in the TinyDB instance
tables = db.tables()
print(tables)

In [ ]:
from tinydb import TinyDB, Query
from pprint import pprint
from difflib import get_close_matches
Object = Query()

db = TinyDB('gamedata.json')

resource_table = db.table('resource')

# Fetch all items from the body table
body_table = db.table('body')
bodies = body_table.all()

# Iterate through each item in the body table
for body in bodies:
  if body.get('name') == 'Resource':
    # Fetch the corresponding resource based on referenceId
    resource = resource_table.get(Object.id == body.get('referenceId'))
    if resource:
      # Update the color of the body to match the resource's color
      body_table.update({'color': resource['color']}, doc_ids=[body.doc_id])

# Print the updated content of the body table to verify the changes
pprint(body_table.all())

In [ ]:

from tinydb import TinyDB, Query

from pprint import pprint
from tinydb.operations import delete

db = TinyDB('gamedata.json')
Object = Query()

# Add the test table to the TinyDB instance
test_table = db.table('test')

# Empty the test_table
test_table.truncate()
test_table.insert({})
# db.drop_table('schema')
# Print the tables to verify the addition
pprint(test_table.all())
print(db.tables())
db.close()

In [ ]:
import shutil

# Copy the gamedata.json file to gamedata.bak
shutil.copy('gamedata.json', 'gamedata.json.bak')

print("gamedata.json has been copied to gamedata.json.bak")

In [ ]:
import shutil

# Restore the gamedata.json file from gamedata.json.bak
shutil.copy('gamedata.json.bak', 'gamedata.json')

print("gamedata.json has been restored from gamedata.json.bak")